<a href="https://colab.research.google.com/github/jmishra01/AI-ML/blob/main/Building_An_Embedding_Model_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Building an Embedding Model From Scratch
========================================

This is a minimal but complete example of how real sentence-embedding models (SBERT, E5, BGE, etc.) are conceptually trained:

1. A small transformer ENCODER ( not a generative decorder) turns a sequence of tokens into a single fixed-size vector (via pooling).
2. We train it with a CONTRASTIVE LOSS: pull unrelated ones apart.
3. At inference time, one forward pass gives you one vector. Compare vectors with cosine similarity for search, clustering, dedup, etc.

This is intentionally tiny (toy vocab, toy data, few params) so it runs fast on CPU and is easy to read end-to-end. Swap in a real tokenizer, a bigger transformer, and millions of sentence pairs to get something production-grade (that's literally what SBERT-style training does).

In [50]:
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

In [51]:
torch.manual_seed(0)
random.seed(0)

## Toy Data
Pairs of sequences that mean the same thing (positives). In real training this is millions of pairs (e.g. question/answer, or two paraphrases, or (query, relevant_document) pairs).

In [52]:
pairs = [
    ("how do i reset my password", "steps to recover a forgotten password"),
    ("best pizza place near me", "good pizza restaurants nearby"),
    ("what is the weather today", "current weather conditions now"),
    ("cheap flights to paris", "affordable airfare to paris"),
    ("symptoms of the flu", "signs that you have influenza"),
    ("how to bake bread at home", "homemade bread baking instructions"),
    ("best laptop for programming", "good laptops for software development")
]

Build a tiny vocabulary from the tiny corpus.

In [53]:
vocab = {"<pad>": 0, "<unk>": 1}
for a, b in pairs:
    for sent in (a, b):
        for word in sent.split():
            if word not in vocab:
                vocab[word] = len(vocab)

print(f"Vocab size: {len(vocab)}")
for k, v in list(vocab.items())[:5]:
    print(f"|{k:>14} | {v:>5} |")

Vocab size: 55
|         <pad> |     0 |
|         <unk> |     1 |
|           how |     2 |
|            do |     3 |
|             i |     4 |


In [54]:
def encode_tokens(sentence, max_len=12):
    ids = [vocab.get(w, vocab["<unk>"]) for w in sentence.split()]
    ids = ids[:max_len] + [vocab["<pad>"]] * max(0, max_len - len(ids))
    return torch.tensor(ids)

## Encoder Itself

$$ \fbox{embedding layer} + \fbox{a couble of self-attention (transformer) blocks} + \fbox{mean-pooling over tokens} → \fbox{one vector} $$


This pooled vector is the sentence embedding.

In [55]:
class TinyTransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2, max_len=12) -> None:
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=128,
            batch_first=True,
            activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(encoder_layer=encoder_layer, num_layers=n_layers)
        self.out_proj = nn.Linear(d_model, d_model) # final embedding projection

    def forward(self, token_ids):
        # token_ids: (batch, seq_len)

        position = torch.arange(token_ids.size(1), device=token_ids.device)
        x = self.token_emb(token_ids) + self.pos_emb(position)

        pad_mask = token_ids == 0
        x = self.encoder(x, src_key_padding_mask=pad_mask)

        # Mean-pool over real (non-pad) tokens -> fixed-size sentence vector
        mask = (~pad_mask).unsqueeze(-1).float()
        summed = (x * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        pooled = summed / counts

        emb = self.out_proj(pooled)

        return F.normalize(emb, p=2, dim=-1)    # unit-length vectors -> cosine sim = dot product


In [56]:
model = TinyTransformerEncoder(vocab_size=len(vocab))
model.eval()

TinyTransformerEncoder(
  (token_emb): Embedding(55, 64, padding_idx=0)
  (pos_emb): Embedding(12, 64)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (out_proj): Linear(in_features=64, out_features=64, bias=True)
)

In [57]:
# install torchinfo
%pip install torchinfo

In [58]:
from torchinfo import summary
summary(model)

Layer (type:depth-idx)                                            Param #
TinyTransformerEncoder                                            --
├─Embedding: 1-1                                                  3,520
├─Embedding: 1-2                                                  768
├─TransformerEncoder: 1-3                                         --
│    └─ModuleList: 2-1                                            --
│    │    └─TransformerEncoderLayer: 3-1                          33,472
│    │    └─TransformerEncoderLayer: 3-2                          33,472
├─Linear: 1-4                                                     4,160
Total params: 75,392
Trainable params: 75,392
Non-trainable params: 0

In [59]:

total_params = sum(i.numel() for i in model.parameters())
print(f'Total params: {total_params}')

Total params: 75392


## Contrastive loss (InfoNCE-style)
For each anchor sentence, its paired sentence should be the closest match among everything else in the batch. his is exactly how SimCSE / SBERT-style training pulls positives together and pushes negatives (everything else in the batch) apart -- no manually labeled negatives needed.

In [60]:
def contrastive_loss(emb_a, emb_b, temperature=0.05):
    # similarity matrix: every anchor vs every candidate in the batch
    logits = emb_a @ emb_b.T / temperature
    labels = torch.arange(emb_a.size(0))    # the correct match is the diagonal
    loss_a = F.cross_entropy(logits, labels)
    loss_b = F.cross_entropy(logits.T, labels)  # symmetric
    return (loss_a + loss_b) / 2


## Train

In [61]:
model = TinyTransformerEncoder(vocab_size=len(vocab))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

a_batch = torch.stack([encode_tokens(a) for a, _ in pairs])
b_batch = torch.stack([encode_tokens(b) for _, b in pairs])

print("Training tiny embedding model... \n")
for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    emb_a = model(a_batch)
    emb_b = model(b_batch)

    loss = contrastive_loss(emb_a=emb_a, emb_b=emb_b)
    loss.backward()
    optimizer.step()

    if epoch % 40 == 0 or epoch == 199:
        print(f"epoch {epoch:3d} loss {loss.item():.4f}")

Training tiny embedding model... 

epoch   0 loss 2.7500
epoch  40 loss 0.0001
epoch  80 loss 0.0001
epoch 120 loss 0.0000
epoch 160 loss 0.0000
epoch 199 loss 0.0000


## Inference
embedd new sentences and compare with cosine similarity (which is just a dot produt, since vectors are unit-normalized).

In [62]:
model.eval()

TinyTransformerEncoder(
  (token_emb): Embedding(55, 64, padding_idx=0)
  (pos_emb): Embedding(12, 64)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (out_proj): Linear(in_features=64, out_features=64, bias=True)
)

In [63]:
def embed(sentence):
    with torch.no_grad():
        return model(encode_tokens(sentence).unsqueeze(0))[0]

In [64]:
print("\n--- Similarity check on TRAINING pairs (should be high) ---")
for a,b in pairs[:3]:
    sim = (embed(a) @ embed(b)).item()
    print(f"{sim:3f} | '{a}' <-> '{b}'")


--- Similarity check on TRAINING pairs (should be high) ---
0.886249 | 'how do i reset my password' <-> 'steps to recover a forgotten password'
0.851306 | 'best pizza place near me' <-> 'good pizza restaurants nearby'
0.924074 | 'what is the weather today' <-> 'current weather conditions now'


A tiny "knowledge base" built only from words our toy model has seen (in a real system this would be thousands of arbitrary document chunks)

In [65]:
knowledge_base = [
    "steps to recover a forgotten password",
    "good pizza restaurants nearby",
    "current weather conditions now",
    "quickest way to learn python programming",
    "affordable airfare to paris",
    "signs that you have influenza",
    "homemade bread baking instructions",
    "good laptops for software development",
]

Step 1: embed knowledge base once

In [66]:
kb_embeddings = torch.stack([embed(doc) for doc in knowledge_base])
kb_embeddings[0]

tensor([-0.2049,  0.0111, -0.0574, -0.0771,  0.0476, -0.1388, -0.0585,  0.1922,
         0.1768, -0.2514, -0.0248, -0.1841,  0.0019, -0.0405, -0.2209,  0.0442,
        -0.0055, -0.0565,  0.0125,  0.0862, -0.0835,  0.0085,  0.0391,  0.0210,
         0.4155,  0.0480,  0.0594,  0.1288,  0.0029, -0.1212, -0.0831,  0.0007,
        -0.0285, -0.0100, -0.1862, -0.2427, -0.1779,  0.2453, -0.1359,  0.1006,
        -0.1981,  0.0614,  0.1916, -0.0447, -0.0027, -0.0155,  0.0901,  0.0228,
        -0.0433, -0.2146, -0.0645,  0.0290, -0.1574, -0.0672,  0.1143,  0.0861,
         0.0069,  0.0198,  0.0175, -0.1059, -0.0073,  0.1449, -0.1072,  0.1272])

In [67]:
def search(query, top_k=2):
    q_emb = embed(query)
    sims = kb_embeddings @ q_emb  # cosine similarity (vectors are unit-norm)
    top_indices = torch.topk(sims, k=top_k).indices
    return [(knowledge_base[i], sims[i].item()) for i in top_indices]


In [68]:
# Query
queries = [
    "what is the weather today",
    "how do i reset my password",
    "best pizza place near me",
]

# Retriever
for query in queries:
    print(f"Query: '{query}'")
    for doc, score in search(query):
        print(f"  {score:.3f}  '{doc}'")
    print()

Query: 'what is the weather today'
  0.924  'current weather conditions now'
  0.269  'steps to recover a forgotten password'

Query: 'how do i reset my password'
  0.886  'steps to recover a forgotten password'
  0.257  'good laptops for software development'

Query: 'best pizza place near me'
  0.851  'good pizza restaurants nearby'
  0.246  'good laptops for software development'

